# Data Parallel Training with DDP

DistributedDataParallel (DDP) is the standard way to train across multiple GPUs in PyTorch. The idea is to make a full copy of your model on each GPU, split the data between them, and after each backward pass have all GPUs average their gradients together. Since every GPU always has the same gradients, the model weights stay in sync.

With 2 GPUs each processing 128 images per step (256 total), the effective batch size doubles and training is roughly 2x faster.

## Imports and Constants

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.multiprocessing as mp
import matplotlib.pyplot as plt
import random
import time
import os

# Same stuff as before
BATCH_SIZE = 512
EPOCHS = 100
LEARNING_RATE = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
MODEL_WEIGHTS_PATH = 'cifar_cnn.pth'
CIFAR_MEAN = (0.49139968, 0.48215827, 0.44653124)
CIFAR_STD  = (0.24703233, 0.24348505, 0.26158768)
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

## Model Architecture: CIFARNet

Same model as the single GPU notebook. DDP doesn't change the model definition at all. It just wraps it after creation to handle gradient synchronization.

In [ ]:
# all networks must extend the nn.Module
class CIFARNet(nn.Module):
    def __init__(self):
        super().__init__()
        # networks must define a constructor to create all of the layers that the network will use

        # our network has 4 convolution layers with 3 mlp layers
        # Our original images have a size of 3x32x32 (3 for rgb then 32x32 image sizes)
        # so we can view each image as having 3 channels i.e 3 seperate images
        # and we convolute over that with 64 different kernels to create 64 new images
        # the convulution matrix will look over all input channels at once so it will take all rgb into account in one go
        # we then create several more convolution layers increasing the number of channels each time
        # kernel_size 3 means we convolute our images with a 3x3 matrix, padding 1 means we make our image 1 pixel bigger on each side
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, padding=1)

        # max pool is an operation that just takes the max from a 2x2 grid
        # it is used to reduce the size of images for faster processing without missing key details
        self.pool = nn.MaxPool2d(2, 2)

        # Batch norm normalizes an entire input to the layer of a neural network to keep all activations reasonable
        # we batch normalize for each convolution layer
        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(128)
        self.bn3 = nn.BatchNorm2d(256)
        self.bn4 = nn.BatchNorm2d(512)

        # dropout means to randomly turn off half the nodes to prevent overfitting
        # this is only used during the training phase and not during testing
        self.dropout = nn.Dropout(p=0.5)

        # the final step in our network is the full connected layers of which we have 3 of
        # We have as input to the first linear layer 512 * 8 * 8 nodes since the last layer of convolution
        # has dimention (B, 512, 8, 8) so that allows one node for each pixel/channel
        # the last layer has 10 nodes since there are 10 different classes we want to identify
        self.fc1 = nn.Linear(512 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        # the forward method defines how data is passed through the layers we created in init

        # for our convolution layers we follow the same basic pattern
        # convlute the input tensor (the raw image)
        # max pool it to reduce the size
        # batch normalize
        # relu (activation function) max(0, x)
        # starting data size is (B, 3, 32, 32)
        x = self.conv1(x) # (B, 64, 32, 32) is the output where B is batch size
        x = self.pool(x)  # (B, 64, 16, 16) output from pool, notice the size of our data is cut in 1/4
        x = self.bn1(x)   # batch normalization doesn't change the size of the data just normalizes the activations
        x = F.relu(x)     # (B, 64, 16, 16) doesn't change the size of the data

        x = self.conv2(x) # (B, 128, 16, 16)
        x = self.pool(x)  # (B, 128, 8, 8) notice size of data cut in 1/4 again
        x = self.bn2(x)
        x = F.relu(x)     # (B, 128, 8, 8)

        x = self.conv3(x) # (B, 256, 8, 8)
        # we don't pool here since our image is already now pretty small
        x = self.bn3(x)
        x = F.relu(x)

        x = self.conv4(x) # (B, 512, 8, 8)
        x = self.bn4(x)
        x = F.relu(x)

        # final shape of data after all convolutions (B, 512, 8, 8)

        # to prepare to put our tensore in the neural network we must flatten it
        # the 1 means to start flattening with the first dimension so only the 512, 8, 8 are flattened
        # this is critical because without that we would flatten all batches into one giant tensore which would be useless
        x = torch.flatten(x, 1) # (B, 512 * 8 * 8) = (B, 32768) output of flatten

        # now we can pass through our flattened tensore into the fully connected layer and another relu activation
        x = self.fc1(x) # (B, 512)
        x = F.relu(x)

        # randomly turn some of the values to 0 to prevent overfitting
        x = self.dropout(x)
        x = self.fc2(x) # (B, 256)
        x = F.relu(x)

        x = self.dropout(x)
        # pass through last fc layer but do not relu because we will softmax later to get probabilities for each class
        x = self.fc3(x) # (B, 10) final size of output

        return x

## Data Loading for DDP

Instead of `shuffle=True` in the DataLoader, DDP uses a `DistributedSampler` that divides the dataset into non-overlapping chunks — one chunk per GPU. Each GPU only loads and processes its own chunk per epoch.

The sampler takes an epoch number (`set_epoch`) to ensure different shuffles each epoch while keeping all ranks in sync. Without it every epoch every gpu would shuffle to the same order.

Rank 0 downloads the dataset first. All other ranks wait at a barrier before trying to read from disk.

In [ ]:
def get_cifar_data_transforms():
    # create a set of augmentations we want to do to our training and data set.
    # for the training set we randomly flip the image to better generalize
    # and also randomly crop out a small portion of it
    # then we normalize the rgb values to be ~N(0,1)
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
    ])
    # We do not augment the data if testing our classifier because we do not want to hinder the model in any way
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
    ])
    return transform_train, transform_test


def get_cifar_data_sets(transform_train, transform_test):
    train_set = datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform_train,
    )
    test_set = datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform_test,
    )
    return train_set, test_set


def cifar10_loaders_ddp(world_size, rank, batch_size=BATCH_SIZE):
    transform_train, transform_test = get_cifar_data_transforms()

    if rank == 0:
        train_set, test_set = get_cifar_data_sets(transform_train, transform_test)

    # Wait for rank 0 to finish downloading
    if world_size > 1:
        dist.barrier()

    # Now all other ranks can safely load the downloaded data
    if rank != 0:
        train_set, test_set = get_cifar_data_sets(transform_train, transform_test)

    # Create DistributedSampler for training data
    # This divides the dataset among processes and handles shuffling
    # shuffle=True means data is shuffled before dividing among processes
    train_sampler = DistributedSampler(
        train_set,
        num_replicas=world_size,
        rank=rank,
        shuffle=True
    )

    # Create DistributedSampler for test data
    # shuffle=False for test data to ensure consistent evaluation
    test_sampler = DistributedSampler(
        test_set,
        num_replicas=world_size,
        rank=rank,
        shuffle=False
    )

    # Create DataLoaders with DistributedSampler
    # When using DistributedSampler, set shuffle=False in DataLoader
    # The sampler handles shuffling instead
    # pin_memory=True speeds up data transfer to GPU
    train_loader = DataLoader(
        train_set,
        batch_size=batch_size,
        sampler=train_sampler,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_set,
        batch_size=batch_size,
        sampler=test_sampler,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    return train_loader, test_loader

## DDP Training Worker

Each process runs this function. It receives its rank (0 or 1) telling it which GPU to use, and does everything a normal training loop does except:
1. The model is wrapped in `DDP`, which automatically syncs gradients across GPUs after each `backward()`
2. Only rank 0 prints output and saves weights
3. We use an output queue to collect print messages — child process stdout doesn't show directly in Jupyter cells

The processes talk to each other over NCCL (NVIDIA's GPU communication library). We tell them where to find each other by specifying a hostname and port before initializing the process group.

In [ ]:
# %%writefile ddp_worker.py

# import torch
# import torch.distributed as dist
# import torch.nn as nn
# import torch.optim as optim
# from torch.nn.parallel import DistributedDataParallel as DDP
# import time

def ddp_worker(rank, world_size):
    # tells the process that the default gpu is rank i.e 0 or 1
    # so every process is using a different gpu
    torch.cuda.set_device(rank)
    # Normally when device is "cuda" this actually gets translated to "cuda:0"
    # which just means use the first GPU. With our multiple GPU setup:
    device = torch.device(f'cuda:{rank}')

    # initialize the NCCL process group so all gpus can talk to each other
    # NCCL backend is optimized for NVIDIA GPUs and is the fastest for multi-GPU training
    # tcp://127.0.0.1:12355 is the address all processes use to find each other on startup
    dist.init_process_group(
        backend='nccl',
        init_method='tcp://127.0.0.1:12355',
        rank=rank,
        world_size=world_size,
        device_id=device
    )

    if rank == 0:
        print(f'Training with DDP on {world_size} GPUs')
    print(f'[rank {rank}] Using device: {device}')

    train_loader, test_loader = cifar10_loaders_ddp(world_size, rank)

    # Create model and move it to the correct GPU
    model = CIFARNet().to(device)

    # Wrap model with DDP
    # This handles gradient synchronization across processes
    # device ids is what gpu this ddp model must explicitly deal with and run through
    # we must specify it so each gpu gets its own model
    model = DDP(model, device_ids=[rank])

    # Loss function and optimizer same as before
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY
    )
    # automatically changes the learning rate to learn faster at the beginning and then more precise later
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    for epoch in range(EPOCHS):
        if rank == 0:
            print(f'\nEpoch {epoch + 1}/{EPOCHS}')
            start_time = time.time()

        # Set epoch for DistributedSampler to ensure different shuffling each epoch
        # without it on every epoch every gpu would have the same random sample
        train_loader.sampler.set_epoch(epoch)

        # Train
        model.train()
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_function(outputs, labels)

            # DDP automatically synchronizes gradients during backward()
            loss.backward()
            optimizer.step()

            _, predicted = torch.max(outputs, 1)
            train_correct += (predicted == labels).sum().item()
            train_total += labels.size(0)

        train_accuracy = 100.0 * train_correct / train_total
        # update our optimizer's learning rate
        scheduler.step()

        if rank == 0:
            print(f'  Training Accuracy: {train_accuracy:.2f}%')

        # Validate
        model.eval()
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                test_correct += (predicted == labels).sum().item()
                test_total += labels.size(0)

        test_accuracy = 100.0 * test_correct / test_total

        if rank == 0:
            print(f'  Test Accuracy:     {test_accuracy:.2f}%')
            print(f'  Epoch time:        {(time.time() - start_time):.2f}s')

    # Save model (only from rank 0 to avoid conflicts)
    # model.module is the underlying CIFARNet — DDP wraps it in model.module
    if rank == 0:
        torch.save(model.module.state_dict(), MODEL_WEIGHTS_PATH)
        print(f'\nSaved model weights to {MODEL_WEIGHTS_PATH}')

    # clean up the process group
    dist.destroy_process_group()

print("ddp worker loaded")

## Launch DDP Training

`mp.spawn` starts one copy of `ddp_worker` per GPU, passing `rank` automatically as the first argument. It blocks until all workers finish, then we drain the output queue and print everything in order.

In [ ]:


assert torch.cuda.device_count() >= 2, (
    'This notebook requires 2 GPUs. '
    'Go to Settings -> Accelerator -> GPU T4 x2 and re-run.'
)

# clean up any stale process groups from a previous run
if dist.is_initialized():
    dist.destroy_process_group()

# how many cuda devices are ther
world_size = torch.cuda.device_count()
print(f'Found {world_size} GPU(s)')

# we using multiprocessing to start a bunch of prcesses that each run our ddp worker
with mp.Manager() as manager:
    mp.start_processes(ddp_worker, args=(world_size,), nprocs=world_size, join=True, start_method='fork')

## Inference

Inference always runs on a single device so no DDP is needed. Load the weights saved by rank 0 and test on a random image.

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# load the trained model and put it on the device we are using
model_inf = CIFARNet().to(device)
model_inf.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
model_inf.eval()

# get our test data set
transform_train, transform_test = get_cifar_data_transforms()
_, test_set = get_cifar_data_sets(transform_train, transform_test)

# pick a random test image
idx = random.randint(0, len(test_set) - 1)
image, label = test_set[idx]

# save unnormalized copy for display — reverse the normalization so colors look right
unnorm = image.clone()
for t, m, s in zip(unnorm, CIFAR_MEAN, CIFAR_STD):
    t.mul_(s).add_(m)
np_img = unnorm.permute(1, 2, 0).numpy()

# run inference
image_batch = image.unsqueeze(0).to(device)
with torch.no_grad():
    output = model_inf(image_batch)
    predicted_idx = output.argmax(dim=1).item()

predicted_class = CLASSES[predicted_idx]
true_class = CLASSES[label]

print('------- Inference Result -------')
print(f'Random Test Index: {idx}')
print(f'Predicted class:   {predicted_class}')
print(f'True label:        {true_class}')
print('--------------------------------')

plt.figure(figsize=(4, 4))
plt.imshow(np_img)
plt.title(f'Predicted: {predicted_class}\nActual: {true_class}', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()